In [1]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import os
import pprint
import time
import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

## get the time period we care about
start_date_data = '2018-01-01'
end_date_data = '2022-12-31'


In [2]:

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
region = md_cooridor.geometry().bounds()

## get the precip, swe, and temp data
precip_dat = ee.ImageCollection("NASA/ORNL/DAYMET_V4") \
                .filterDate(start_date_data, end_date_data) \
                .select('prcp', 'swe', 'tmax', 'tmin') \
                .map(lambda image: image.clip(region))

## export the images
def export_images_by_year(collection, start_year, end_year, region):
    for year in range(start_year, end_year + 1):
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"
        
        yearly_collection = collection.filterDate(start_date, end_date)

        # Generate a list of dates within the year to create individual exports
        dates = yearly_collection.aggregate_array('system:time_start').getInfo()
        
        for date_millis in dates:
            date = ee.Date(date_millis)
            image = yearly_collection.filterDate(date, date.advance(1, 'day')).first()
            date_str = date.format('YYYY-MM-dd').getInfo()
            
            # Define the export parameters
            export_params = {
                'description': f'precip_data_{year}_{date_str}',
                'image': image,
                'scale': 500,  # Adjust according to your needs
                'folder': 'GEE_Yearly_Exports',
                'region': region.getInfo()['coordinates'],
                'fileFormat': 'GeoTIFF',
                'maxPixels': 1e13
            }
            
            # Start the export task for each image
            task = ee.batch.Export.image.toDrive(**export_params)
            task.start()
            print(f"Export task started for {year} on {date_str}")


# Execute the export function
export_images_by_year(precip_dat, 2018, 2022, region)


Export task started for 2018 on 2018-01-01
Export task started for 2018 on 2018-01-02
Export task started for 2018 on 2018-01-03
Export task started for 2018 on 2018-01-04
Export task started for 2018 on 2018-01-05
Export task started for 2018 on 2018-01-06
Export task started for 2018 on 2018-01-07
Export task started for 2018 on 2018-01-08
Export task started for 2018 on 2018-01-09
Export task started for 2018 on 2018-01-10
Export task started for 2018 on 2018-01-11
Export task started for 2018 on 2018-01-12
Export task started for 2018 on 2018-01-13
Export task started for 2018 on 2018-01-14
Export task started for 2018 on 2018-01-15
Export task started for 2018 on 2018-01-16
Export task started for 2018 on 2018-01-17
Export task started for 2018 on 2018-01-18
Export task started for 2018 on 2018-01-19
Export task started for 2018 on 2018-01-20
Export task started for 2018 on 2018-01-21
Export task started for 2018 on 2018-01-22
Export task started for 2018 on 2018-01-23
Export task